## TEST SERPENT -> IR

In [1]:
from pathlib import Path
import src.serpent_mat_ir as sir
import importlib
importlib.reload(sir)

input="2D-CANDU"

lines = sir.load_serpent_lines(Path(input))
therm = sir.parse_therm_cards(lines)
materials_ir = sir.parse_materials_to_ir(lines, therm)

/Users/jenschristiansen/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/data/thermal.py:202: UserWarning: Thermal scattering material "lwj3" is not recognized. Assigning a name of c_H_in_H2O.
  warn('Thermal scattering material "{}" is not recognized. '
/Users/jenschristiansen/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/data/thermal.py:202: UserWarning: Thermal scattering material "hwj3" is not recognized. Assigning a name of c_D_in_D2O.
  warn('Thermal scattering material "{}" is not recognized. '


In [2]:
materials_ir = sir.parse_materials_to_ir(lines, therm, preprocess=False)
print(len(materials_ir), list(materials_ir.keys())[:20])

6 ['fuel', 'clad', 'tube', 'caltube', 'cool', 'moder']


In [3]:
mat_names = [l.split()[1] for l in lines if l.split() and l.split()[0].lower() == "mat"]
print("mat_names:", len(mat_names), mat_names[:10])
print("parsed keys:", len(materials_ir), list(materials_ir.keys())[:10])

mat_names: 6 ['fuel', 'clad', 'tube', 'caltube', 'cool', 'moder']
parsed keys: 6 ['fuel', 'clad', 'tube', 'caltube', 'cool', 'moder']


In [4]:
# TEST IF MATERIAL(S) ARE CORRECT
materials_ir[mat_names[0]]

{'type': 'material',
 'name': 'fuel',
 'material_id': None,
 'density': 10.437501,
 'density_units': 'g/cm3',
 'density_mode': 'mass',
 'temperature': None,
 'sab': [],
 'keywords': {},
 'nuclides': [{'token': '8016.09c',
   'zaid': '8016',
   'name': 'O16',
   'element': 'O',
   'A': 16,
   'percent': -11.8473,
   'percent_mode': 'wo',
   'metastable_ignored': False},
  {'token': '92235.09c',
   'zaid': '92235',
   'name': 'U235',
   'element': 'U',
   'A': 235,
   'percent': -0.627118,
   'percent_mode': 'wo',
   'metastable_ignored': False},
  {'token': '92238.09c',
   'zaid': '92238',
   'name': 'U238',
   'element': 'U',
   'A': 238,
   'percent': -87.5256,
   'percent_mode': 'wo',
   'metastable_ignored': False}]}

In [5]:
## TEST IF GEOMETRY IS CORRECT
from src.serpent_geom_ir import geometry_ir_from_file

geom_ir = geometry_ir_from_file(input)
print(geom_ir["surfaces"])
print(geom_ir["pins"])

{'1': {'type': 'surface', 'name': '1', 'surface_type': 'cyl', 'coefficients': [0.0, 0.0, 5.1689]}, '2': {'type': 'surface', 'name': '2', 'surface_type': 'cyl', 'coefficients': [0.0, 0.0, 5.6032]}, '3': {'type': 'surface', 'name': '3', 'surface_type': 'cyl', 'coefficients': [0.0, 0.0, 6.4478]}, '4': {'type': 'surface', 'name': '4', 'surface_type': 'cyl', 'coefficients': [0.0, 0.0, 6.5875]}, '5': {'type': 'surface', 'name': '5', 'surface_type': 'sqc', 'coefficients': [0.0, 0.0, 14.287]}}
{'1': {'type': 'pin', 'name': '1', 'layers': [{'fill': 'fuel', 'fill_mode': 'material', 'radius': 0.6122}, {'fill': 'clad', 'fill_mode': 'material', 'radius': 0.654}, {'fill': 'cool', 'fill_mode': 'material', 'radius': None}]}}


In [6]:
# Geometry IR summary
counts = {k: len(v) for k, v in geom_ir.items() if isinstance(v, dict)}
print(counts)

print("Pins:", sorted(geom_ir["pins"].keys()))
print("Lattices:", sorted(geom_ir["lattices"].keys()))

{'surfaces': 5, 'cells': 6, 'pins': 1, 'lattices': 1, 'settings': 1}
Pins: ['1']
Lattices: ['10']


In [7]:
# INSPECT A LATTICE
geom_ir["lattices"]

{'10': {'type': 'lattice', 'name': '10', 'lat_type': '4', 'lat_type_int': 4}}

## TEST IR -> OpenMC

In [8]:
from pathlib import Path
from src.serpent_mat_ir import load_serpent_lines, parse_therm_cards, parse_materials_to_ir
from src.ir_openmc import materials_from_ir

lines = load_serpent_lines(Path("SPX"))
therm = parse_therm_cards(lines)
materials_ir = parse_materials_to_ir(lines, therm)

openmc_materials = materials_from_ir(materials_ir)
print(len(openmc_materials), list(openmc_materials.keys())[:10])
openmc_materials["wrap"]

170 ['MAT_01I01', 'MAT_02I01', 'MAT_03I01', 'MAT_04I01', 'MAT_05I01', 'MAT_06I01', 'MAT_07I01', 'MAT_08I01', 'MAT_09I01', 'MAT_10I01']


Material
	ID             =	160
	Name           =	wrap
	Temperature    =	673.0
	Density        =	None [sum]
	Volume         =	None [cm^3]
	Depletable     =	False
	S(a,b) Tables  
	Nuclides       
	C12            =	0.000193347106986 [ao]
	C13            =	2.1658930139999997e-06 [ao]
	Si28           =	0.000928063  [ao]
	Si29           =	4.552e-05    [ao]
	Si30           =	2.90426e-05  [ao]
	P31            =	4.54479e-05  [ao]
	Ti46           =	3.36969e-05  [ao]
	Ti47           =	2.97418e-05  [ao]
	Ti48           =	0.000288578  [ao]
	Ti49           =	2.07448e-05  [ao]
	Ti50           =	1.94665e-05  [ao]
	Cr50           =	0.000653123  [ao]
	Cr52           =	0.0121112    [ao]
	Cr53           =	0.00134737   [ao]
	Cr54           =	0.000329182  [ao]
	Mn55           =	0.00145198   [ao]
	Fe54           =	0.0032908    [ao]
	Fe56           =	0.0498158    [ao]
	Fe57           =	0.00113025   [ao]
	Fe58           =	0.000147824  [ao]
	Ni58           =	0.00771919   [ao]
	Ni60           =	0.0028744    [ao

In [9]:
type(openmc_materials["wrap"])

openmc.material.Material

In [10]:
from pathlib import Path
import openmc
from src.serpent_to_openmc import build_openmc_components
import os 

materials, geom, root = build_openmc_components(Path('2D-BWR'))
print('materials', len(materials))
print('cells', len(geom['cells']))
print('lattices', len(geom['lattices']))

# Edit materials/geometry before assembling the model
model = openmc.Model(geometry=openmc.Geometry(root))
model.materials = openmc.Materials(materials.values())

xs_path = "/Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/"
model.materials.cross_sections = xs_path+"cross_sections.xml"

# Add a tally
tally = openmc.Tally(name="flux")
tally.filters = [openmc.CellFilter(list(geom["cells"].values()))]
tally.scores = ["flux"]
model.tallies = openmc.Tallies([tally])

materials 12
cells 6
lattices 1


In [18]:
from pathlib import Path
import openmc
from src.serpent_to_openmc import build_openmc_components

# Optional: set this if you don't rely on OPENMC_CROSS_SECTIONS env var
# openmc.config['cross_sections'] = '/absolute/path/to/cross_sections.xml'

materials, geom, root = build_openmc_components(Path('2D-BWR'))
model = openmc.Model(geometry=openmc.Geometry(root))
model.materials = openmc.Materials(materials.values())

xs_path = "/Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/"
model.materials.cross_sections = xs_path+"cross_sections.xml"

plot = openmc.Plot()
plot.basis = 'xy'
plot.origin = (0.0, 0.0, 0.0)
plot.width = (40.0, 40.0)
plot.pixels = (800, 800)
plot.color_by = 'cell'
plot.filename = 'bwr_xy'
model.plots = openmc.Plots([plot])
model.plot_geometry()


# settings
settings = openmc.Settings()
settings.batches = 40
settings.inactive = 10
settings.particles = 2000
# Allow interpolation between library temperatures
settings.temperature = {
    "method": "interpolation",
    "range": (250.0, 2500.0),
}

model.settings = settings

# set boundary conditions
geom["surfaces"]["3"].boundary_type = "reflective"


# optional: set a simple point source if needed
model.settings.source = openmc.Source(space=openmc.stats.Point((0.0, 0.0, 0.0)))

model.run()

/Users/jenschristiansen/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/data/thermal.py:202: UserWarning: Thermal scattering material "lwj3" is not recognized. Assigning a name of c_H_in_H2O.
  warn('Thermal scattering material "{}" is not recognized. '
/Users/jenschristiansen/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Lattice instance already exists with id=10.
  warn(msg, IDWarning)
/Users/jenschristiansen/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another UniverseBase instance already exists with id=0.
  warn(msg, IDWarning)


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/Users/jenschristiansen/miniconda3/envs/openmc/lib/python3.13/site-packages/openmc/source.py:668: FutureWarning: This class is deprecated in favor of 'IndependentSource'
  warnings.warn(


 Reading U235 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/U235.h5
 Reading U238 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/U238.h5
 Reading O16 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/O16.h5
 Reading Gd154 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/Gd154.h5
 Reading Gd155 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/Gd155.h5
 Reading Gd156 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/Gd156.h5
 Reading Gd157 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/Gd157.h5
 Reading Gd158 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/Gd158.h5
 Reading Gd160 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/Gd160.h5
 Reading Sn112 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/Sn112.h5
 Reading Sn114 from
 /Users/jenschristiansen/Desktop/openmc/data/jeff-3.3-hdf5/Sn114.h5
 Reading Sn115 from
 /Users/jenschristia

PosixPath('/Users/jenschristiansen/Desktop/openmc/openmc_serpent_adapter/statepoint.40.h5')

In [14]:
geom["surfaces"]

{'1': <RectangularPrism at 0x16db14350>,
 '2': <RectangularPrism at 0x16db14290>,
 '3': <RectangularPrism at 0x16daf0d60>,
 '4': <RectangularPrism at 0x16daf12e0>,
 '5': <RectangularPrism at 0x16daae490>}